In [1]:
from dataclasses import dataclass
from typing import Optional
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

print("GROQ:", bool(os.getenv("GROQ_API_KEY")))
print("OPENROUTER:", bool(os.getenv("OPENROUTER_API_KEY")))

@dataclass(frozen=True)
class Provider:
    """ One Provider to reliably route request across all inference providers"""
    name: str
    env_var: str
    is_free: bool
    base_url: Optional[str]
    model: Optional[str]

PROVIDERS = [
    Provider("Groq", "GROQ_API_KEY", True, "https://api.groq.com/openai/v1", "openai/gpt-oss-120b"),
    Provider("OpenRouter", "OPENROUTER_API_KEY", True, "https://openrouter.ai/api/v1", "openrouter/free"),
]

def select_provider() -> Provider:
    for provider in PROVIDERS:
        if(os.getenv(provider.env_var)):
            return provider;

    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider API Key is set add one of the {expected} to your environment variables")


def build_client(provider: Provider) -> OpenAI:
    api_key = os.getenv(provider.env_var)
    if provider.base_url is None:
        return OpenAI(api_key=api_key)

    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url
    )

def have_any_key() -> bool:
    return any(os.getenv(provider.env_var) for provider in PROVIDERS)


print("Found a provider key" if have_any_key() else "No Provider key found")

GROQ: True
OPENROUTER: True
Found a provider key


In [ ]:
SYSTEM_PROMPT = """
You are a product review sentiment analysis expert.

You are given a product review and you need to analyze the sentiment of the review.

I need you to analyze the review and return the sentiment of the review in a JSON format.

The JSON should have the following fields:
- sentiment: The sentiment of the review.
- confidence: The confidence of the sentiment.
- reasoning: The reasoning for the sentiment.

""".strip()

In [33]:
def analyze_review(review: str) -> str:
    """
    Analyze the sentiment of the review.
    """
    provider = select_provider()
    client = build_client(provider)

    response = client.chat.completions.create(
        model = provider.model,
        max_completion_tokens=100,
        reasoning_effort="low",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": review
            }
        ],
    )

    print(response.choices[0].message.content)

In [34]:
analyze_review("Product is good and the price is reasonable")

{
  "sentiment": "positive",
  "confidence": 0.97
}
